# 📊 Agri Advisory QA Test Log - Column Audit Report

This notebook performs a detailed column-by-column audit on the sheet `Test Log_1` inside `Agri_Advisory_QA_Test_Log (1.0).xlsx` to check compliance against the **Golden Rules** of data consistency, types, and nullability required for the **ACE Executive Health Dashboard**.

In [ ]:
import openpyxl
import datetime
import re
import pandas as pd

xlsx_path = "Agri_Advisory_QA_Test_Log (1.0).xlsx"
wb = openpyxl.load_workbook(xlsx_path, data_only=True)
sheet = wb['Test Log_1']
print(f"Sheet Loaded: {sheet.title} | Max Rows: {sheet.max_row} | Max Columns: {sheet.max_column}")

## 📐 Define Golden Rules & Constraints
Each column has defined types (Nominal, Date, Categorical, Non-Categorical text, Time, Duration) and Nullability rules (Nullable vs Required).

In [ ]:
# Read headers from row 3
headers = []
for c in range(1, sheet.max_column + 1):
    val = sheet.cell(row=3, column=c).value
    headers.append(val.strip().replace('\n', ' ') if val else f"Unnamed: {c}")

# Load valid data rows (skipping empty and project summary rows)
data_rows = []
for r in range(4, sheet.max_row + 1):
    row_vals = [sheet.cell(row=r, column=c).value for c in range(1, len(headers) + 1)]
    if any(v is not None for v in row_vals):
        test_id = str(row_vals[0]).strip() if row_vals[0] is not None else ""
        if test_id and not test_id.startswith("Project:"):
            data_rows.append(row_vals)

print(f"Total data rows parsed for audit: {len(data_rows)}")

## 🔍 Audit Logic
Let's execute the column-by-column audit checking for Null violations, Casing issues, and Format errors.

In [ ]:
audit_results = []

for idx, col_name in enumerate(headers):
    vals = [row[idx] for row in data_rows]
    total_rows = len(vals)
    null_count = sum(1 for v in vals if v is None or str(v).strip() == "" or str(v).strip().upper() == "NAN")
    filled_count = total_rows - null_count
    
    unique_vals = set(str(v).strip() for v in vals if v is not None and str(v).strip() != "")
    lowercase_unique = set(str(v).strip().lower() for v in vals if v is not None and str(v).strip() != "")
    casing_issues = len(unique_vals) - len(lowercase_unique)
    
    # Classify column rule rules
    col_rule_type = "Categorical"
    nullable_rule = "Nullable"
    
    if "ID" in col_name:
        col_rule_type = "Nominal"
        nullable_rule = "Required"
    elif "Date" in col_name:
        col_rule_type = "Date"
        nullable_rule = "Nullable"
    elif "Time" in col_name or "TAT" in col_name:
        col_rule_type = "Duration" if ("TAT" in col_name or "Response" in col_name) else "Time"
        nullable_rule = "Nullable"
    elif any(kw in col_name for kw in ["Text", "Remark", "Name", "Description"]):
        col_rule_type = "Non-Categorical"
        nullable_rule = "Required" if "Tester Name" in col_name else "Nullable"
        
    # Check formatting issues
    invalid_formats = 0
    for v in vals:
        if v is None:
            continue
        v_str = str(v).strip()
        if not v_str:
            continue
            
        if col_rule_type == "Date":
            if not isinstance(v, (datetime.datetime, datetime.date)):
                if not re.match(r'^\d{2}[-/]\d{2}[-/]\d{4}$', v_str) and not re.match(r'^\d{2}[-/][a-zA-Z]{3}[-/]\d{2,4}$', v_str):
                    invalid_formats += 1
        elif col_rule_type == "Time":
            if not isinstance(v, (datetime.time, datetime.datetime)):
                if not re.match(r'^\d{2}:\d{2}(:\d{2})?(\s?[aApP][mM])?$', v_str) and not re.match(r'^\d{2}\.\d{2}(\.\d{2})?$', v_str):
                    invalid_formats += 1
        elif col_rule_type == "Duration":
            if not isinstance(v, (int, float, datetime.timedelta)):
                if not re.match(r'^-?\d+(\.\d+)?$', v_str):
                    invalid_formats += 1

    # Check null violation
    null_violation = "YES" if (nullable_rule == "Required" and null_count > 0) else "NO"
    
    audit_results.append({
        "Column Index": idx + 1,
        "Column Name": col_name,
        "Rule Type": col_rule_type,
        "Nullability Spec": nullable_rule,
        "Null Count": null_count,
        "Null Violation?": null_violation,
        "Casing Inconsistencies": casing_issues,
        "Format Errors": invalid_formats,
        "Status": "OK" if (null_violation == "NO" and casing_issues == 0 and invalid_formats == 0) else "Needs Prep"
    })

audit_df = pd.DataFrame(audit_results)
audit_df.to_csv("column_audit_summary.csv", index=False)
print("Audit complete. Summary saved to column_audit_summary.csv")

## 📊 Summary of Audit Findings
Let's display columns that violate rules.

In [ ]:
print("--- Columns with Null Violations (Required columns containing nulls) ---")
print(audit_df[audit_df["Null Violation?"] == "YES"][["Column Name", "Null Count"]].to_string(index=False))

print("\n--- Columns with Format Errors (Invalid format in Date/Time/Duration) ---")
print(audit_df[audit_df["Format Errors"] > 0][["Column Name", "Format Errors"]].to_string(index=False))

print("\n--- Columns with Casing Inconsistencies (Spelling casing duplicate options) ---")
print(audit_df[(audit_df["Casing Inconsistencies"] > 0) & (audit_df["Rule Type"] == "Categorical")][["Column Name", "Casing Inconsistencies"]].to_string(index=False))